In [3]:
import ast
import csv
from distutils.log import error
import json
import math
import numpy as np

import led_config_utils
import mesh_config
from funky_lights import wavefront, led_config

from importlib import reload
reload(mesh_config)

START_OFFSET = 0.0
END_OFFSET = 0.0


def Ry(theta):
    return np.matrix([[math.cos(theta), 0, math.sin(theta)],
                     [0, 1, 0],
                     [-math.sin(theta), 0, math.cos(theta)]])


def createNodesFromCSV(csv_points):
    nodes = None
    prev_node = None
    # Rotate points around Y to match the Funky model orientation
    R = Ry(math.radians(90))
    for point in csv_points:
        point = np.array(point)
        point = (R * point.reshape((3, 1))).reshape((1, 3))
        point = np.squeeze(np.asarray(point))
        node = led_config_utils.Node(p=point)
        if nodes == None:
            nodes = node
        if prev_node:
            prev_node.next = node
        prev_node = node
    return nodes


all_segments = {}
with open('../config/led_config_coat_lara_actual.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)  # skip header
    segments = []

    for row in reader:
        uid = int(row[0])
        name = row[1]
        actual_num_leds = int(row[2])
        actual_length = float(row[3])
        reverse = (row[4] == 'TRUE')
        led_offset = int(row[5])
        sub_component = row[6]
        actual_num_adressable_leds = int(row[7])
        csv_points = ast.literal_eval(row[10])

        nodes = createNodesFromCSV(csv_points)
        if nodes == None:
            print('No nodes available.')
            continue

        modelled_length = led_config_utils.line_segments_length(nodes)
        modelled_length = modelled_length - START_OFFSET - END_OFFSET

        leds_distance = modelled_length / actual_num_adressable_leds
        points = led_config_utils.trace_line_segments(
            nodes, actual_num_adressable_leds, START_OFFSET, leds_distance)
        if len(points) == 0:
            print('No points available.')
            continue

        if reverse:
            points = np.flip(points, axis=0)

        if led_offset > 0:
            points = np.concatenate(
                (points[led_offset:], points[:led_offset]), axis=0)

        segment = led_config.Segment(
            uid=uid, name=name, points=points, num_leds=points.shape[0], length=actual_length)
        print('Segment %s: length=%.1fm, num_leds=%s' %
              (segment.name, segment.length, segment.num_leds))
        segments.append(segment)
        all_segments[uid] = segment

# Collapse some segments
SEGMENT_1 = [3, 9, 11, 10, 4]  # buttons and collar
SEGMENT_2 = [7, 13]  # right side seam
SEGMENT_3 = [8, 14]  # left side seam
SEGMENT_4 = [12]  # hem
SEGMENT_5 = [1, 15, 5]  # front/back seam right
SEGMENT_6 = [2, 16, 6]  # front/back seam left

for merge_list in [SEGMENT_1, SEGMENT_2, SEGMENT_3, SEGMENT_4, SEGMENT_5, SEGMENT_6]:
    merged_segment = all_segments[merge_list[0]]
    for uid in merge_list[1:]:
        segment = all_segments.pop(uid)
        merged_segment.merge(segment)

# Create LED config
config = led_config.LedConfig()
for segment in all_segments.values():
    config.led_segments.append(segment)
    config.total_num_segments += 1
    config.total_length += segment.length
    config.total_num_leds += segment.num_leds

with open('../config/led_config_coat_lara.json', 'w', encoding='utf-8') as f:
    json.dump(config.to_dict(), f, ensure_ascii=False, indent=4)

Segment back_seam_right: length=0.9m, num_leds=50
Segment back_seam_left: length=0.9m, num_leds=50
Segment buttons_right: length=1.0m, num_leds=52
Segment buttons_left: length=1.0m, num_leds=52
Segment front_seam_right: length=0.9m, num_leds=51
Segment front_seam_left: length=0.9m, num_leds=51
Segment side_seam_right: length=0.7m, num_leds=38
Segment side_seam_left: length=0.7m, num_leds=37
Segment colar_right: length=0.1m, num_leds=6
Segment colar_left: length=0.1m, num_leds=6
Segment colar_center: length=0.5m, num_leds=37
Segment hem: length=1.7m, num_leds=110
Segment shoulder_right: length=0.7m, num_leds=36
Segment shoulder_left: length=0.7m, num_leds=36
Segment shoulder_seam_right: length=0.2m, num_leds=11
Segment shoulder_seam_left: length=0.2m, num_leds=11
